# Case RJ - Grupo 2 — Silver: Custos e Pesquisa de Satisfação

Sou o Milton e fico responsável por essas duas tabelas. Leio cada uma da bronze e aplico a limpeza
da camada silver — decido a lógica de cada tabela do jeito que achar melhor, respeitando os pontos
abaixo.

Pontos que vou considerar:
- **Custos**: o campo `custo` vem com formatos mistos (vírgula/ponto, símbolo de moeda, texto colado
  tipo "reais"). Vou ter cuidado especial pra não tratar ponto decimal como separador de milhar
  quando não há vírgula — isso infla o valor várias vezes. Vou checar `id_custo` duplicado antes de
  somar, e guardar o valor como `decimal(12,2)` (não `double`), pra não ter ruído de ponto flutuante
  numa soma de valor em dinheiro.
- **Pesquisa de satisfação**: nota só é válida de 1 a 5 — fora disso (incluindo vazio) vira
  inválida, marcada como flag, nunca descartada em silêncio. Além da nota estrita, vou criar uma
  `nota_ajustada`: `6` vira `5` (parece erro de clique acima do teto da escala) e `-1`/`0` viram `1`
  (parece erro embaixo do piso); quando é vazio ou `"NULL"` literal, fica `NULL` nas duas colunas —
  aqui eu trato como "não teve avaliação", não como nota ruim. As duas colunas ficam na tabela, pra
  ser uma premissa explícita e não uma troca silenciosa da regra que o resto do grupo já usa.

In [ ]:
CATALOGO = "dados_vcredit"
SCHEMA = "default"
ORIGEM_CUSTOS = f"{CATALOGO}.{SCHEMA}.custos_raw"
ORIGEM_PESQUISA = f"{CATALOGO}.{SCHEMA}.pesquisa_satisfacao_raw"
DESTINO_CUSTOS = f"{CATALOGO}.{SCHEMA}.custos_silver"
DESTINO_PESQUISA = f"{CATALOGO}.{SCHEMA}.pesquisa_satisfacao_silver"

## Limpeza — Custos (a preencher)

In [ ]:
from pyspark.sql import functions as F


def _limpa_nomes(df):
    # a bronze preserva o header como veio do arquivo (fiel a origem); aqui corrijo isso
    for antigo in df.columns:
        novo = antigo.replace("\ufeff", "").strip().lower()
        if novo != antigo:
            df = df.withColumnRenamed(antigo, novo)
    return df


df_custos_raw = _limpa_nomes(spark.table(ORIGEM_CUSTOS))

# custo vem em pelo menos 8 formatos (virgula, ponto, "R$ ", sufixo "reais", negativo em qualquer
# combinacao) - ver analises/perfil_custos_pesquisa.md. Mantenho so digitos/virgula/ponto/sinal; se
# tiver virgula, ela e o separador decimal e qualquer ponto e separador de milhar (removo); se nao
# tiver virgula, o ponto que sobrar ja e decimal (nunca remover ponto as ciegas, senao "74.71reais"
# viraria "7471" em vez de 74.71).
_bruto_custo = F.regexp_replace(F.trim(F.col("custo")), r"[^0-9,.\-]", "")
_numero_custo = F.when(
    _bruto_custo.contains(","),
    F.regexp_replace(F.regexp_replace(_bruto_custo, r"\.", ""), ",", "."),
).otherwise(_bruto_custo)

custos_silver = (
    df_custos_raw
    # decimal(12,2) em vez de double - soma de dinheiro nao pode ter ruido de ponto flutuante binario
    .withColumn("custo", _numero_custo.try_cast("decimal(12,2)"))
    .select(
        F.trim("id_custo").alias("id_custo"),
        F.trim("id_chamado").alias("id_chamado"),
        "custo",
    )
    .withColumn("dq_custo_invalido", F.col("custo").isNull())
    .withColumn("dq_custo_nao_positivo", F.col("custo") <= 0)
    # os 781 id_custo duplicados sao sempre a mesma linha repetida (mesmo id_chamado e mesmo custo,
    # nao dois lancamentos diferentes) - dropDuplicates so remove a copia, nao perde informacao.
    .dropDuplicates(["id_custo"])
)

(
    custos_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_CUSTOS)
)

print(f"custos: {df_custos_raw.count():,} -> {spark.table(DESTINO_CUSTOS).count():,} linhas (esperado: 130.288)")
print(f"custo invalido (flag): {spark.table(DESTINO_CUSTOS).where('dq_custo_invalido').count():,}")

## Limpeza — Pesquisa de satisfação (a preencher)

In [ ]:
from pyspark.sql import Window

df_pesquisa_raw = _limpa_nomes(spark.table(ORIGEM_PESQUISA))

# nota_atendimento tem duas formas de "ausente" (campo vazio e a string literal "NULL" - ver
# analises/perfil_custos_pesquisa.md) - try_cast pra int resolve as duas de uma vez, porque "NULL"
# tambem nao converte. Fora isso, so 1-5 e valido (-1/0/6 ficam de fora, nota_i guarda o valor bruto
# pra eu poder calcular a nota_ajustada a seguir).
pesquisa_tipada = df_pesquisa_raw.withColumn("nota_i", F.trim(F.col("nota_atendimento")).try_cast("int"))

# nota_ajustada: 6 -> 5 (passou 1 do teto, parece erro de clique) e -1/0 -> 1 (parece erro embaixo do
# piso); quando nota_i e nulo (vazio ou "NULL" literal) fica nulo tambem - tratado como "nao teve
# avaliacao", nao como nota ruim. E uma premissa nossa, por isso fica numa coluna separada da nota
# estrita (nunca substitui ela).
_nota_ajustada = (
    F.when(F.col("nota_i").isNull(), F.lit(None))
     .when(F.col("nota_i") == 6, F.lit(5))
     .when(F.col("nota_i").isin(-1, 0), F.lit(1))
     .when(F.col("nota_i").between(1, 5), F.col("nota_i"))
     .otherwise(F.lit(None))
)

pesquisa_bruta = (
    pesquisa_tipada
    .select(
        F.trim("id_pesquisa").alias("id_pesquisa"),
        F.trim("id_chamado").alias("id_chamado"),
        F.when(F.col("nota_i").between(1, 5), F.col("nota_i")).alias("nota"),
        _nota_ajustada.alias("nota_ajustada"),
    )
    .withColumn("dq_nota_invalida", F.col("nota").isNull())
)

# ate 2 pesquisas por id_chamado; quando ha duas, priorizo a linha com nota valida (a maioria e so
# a mesma nota repetida, mas em alguns casos uma das duas e invalida e a outra nao).
_janela = Window.partitionBy("id_chamado").orderBy(F.col("nota").isNull().asc(), F.col("id_pesquisa").asc())
pesquisa_silver = (
    pesquisa_bruta.withColumn("_rank", F.row_number().over(_janela))
    .where(F.col("_rank") == 1).drop("_rank")
)

(
    pesquisa_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(DESTINO_PESQUISA)
)

print(f"pesquisa_satisfacao: {df_pesquisa_raw.count():,} -> {spark.table(DESTINO_PESQUISA).count():,} linhas (1 por id_chamado)")
print(f"nota invalida (flag): {spark.table(DESTINO_PESQUISA).where('dq_nota_invalida').count():,}")
print(f"nota_ajustada preenchida: {spark.table(DESTINO_PESQUISA).where('nota_ajustada is not null').count():,}")

## Checklist

- [ ] Confiro que nenhum ponto foi removido indiscriminadamente da regra de custo.
- [ ] Confiro que `custo` ficou como `decimal(12,2)`, não `double`.
- [ ] Confiro que nota fora de 1-5 fica marcada como inválida, não descartada.
- [ ] Confiro que `nota_ajustada` é `NULL` quando a nota é vazia/`"NULL"` (não vira `1`).
- [ ] Rodo duas vezes e vejo se o resultado é o mesmo.